<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/Agents/HuggingFace/PaliGemma_init.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00


In [ ]:
!pip install -q langgraph langchain-core

In [ ]:
import torch
from typing import Dict, Any, TypedDict
from PIL import Image
from langgraph.graph import StateGraph, START, END
from transformers import pipeline
from google.colab import userdata


# ==========================================
# 1. SETUP GRAPH STATE & PALIGEMMA PIPELINE
# ==========================================

# Define the data schema traveling through the graph
class AgentState(TypedDict):
    image_path: str          # Input to graph
    raw_ocr_text: str        # Output of Step 1
    final_output: str        # Output of Step 2

# Load PaliGemma pipeline (using mix-224 checkpoint optimized for Colab VRAM)
hf_token = userdata.get('HF_TOKEN')
model_id = "google/paligemma2-3b-mix-224"

# If you fine-tuned adapters using LoRA, swap model_id out for your local directory path
pipe = pipeline(
    task="image-text-to-text",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 2. DEFINE LANGGRAPH WORKFLOW NODES
# ==========================================

def ocr_node(state: AgentState) -> Dict[str, Any]:
    """Extracts text from image utilizing the fine-tuned vision pipeline."""
    print("--- RUNNING PALIGEMMA OCR ---")

    # Load image from local file path or URL stored in state
    image = Image.open(state["image_path"]).convert("RGB")

    # Execute structural 'ocr' task prefix
    outputs = pipe(image, text="ocr")
    extracted_text = outputs[0]['generated_text']

    # Return updates to merge directly into the global AgentState
    return {"raw_ocr_text": extracted_text}


def clean_node(state: AgentState) -> Dict[str, Any]:
    """Performs post-processing cleanup on the extracted text string."""
    print("--- POST-PROCESSING TEXT ---")
    raw_text = state["raw_ocr_text"]

    # Apply custom parsing logic (strip structural tags, extra newlines, or whitespace)
    cleaned_text = raw_text.strip()

    return {"final_output": cleaned_text}

# ==========================================
# 3. COMPILE THE STATEGRAPH
# ==========================================

# Build the layout configuration
workflow = StateGraph(AgentState)

# Add our custom processing steps to the topology
workflow.add_node("ocr_node", ocr_node)
workflow.add_node("clean_node", clean_node)

# Construct routing architecture
workflow.add_edge(START, "ocr_node")
workflow.add_edge("ocr_node", "clean_node")
workflow.add_edge("clean_node", END)

# Compile into an executable application graph
app = workflow.compile()

In [ ]:
# Provide an image file (Ensure you have this file locally in Colab or replace with an absolute path)
inputs = {"image_path": "/content/images.jpeg"}

# Run through the compiled pipeline
config = {"configurable": {"thread_id": "ocr_session_1"}}
result = app.invoke(inputs, config=config)

print("\n--- RESULTS ---")
print("Extracted Data:", result["final_output"])

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RUNNING PALIGEMMA OCR ---
--- POST-PROCESSING TEXT ---

--- RESULTS ---
Extracted Data: ocrRESIDENTIAL LEASE AGREEMENT
(Single Family Home)
Note: This template was created and copyrighted by You and I can and may only be used for the specific purpose of the document. You may not copy, reproduce, modify, or otherwise use the document or any portion of it for any other purpose without the written permission of You and I.
1. PARTIES
This Residential Rental Agreement ("Agreement") is entered into by and between
(a) "Tenant" and
(b) "Lessor" and
(c) Landlord's Landlord and tenant are referred to in this Agreement as the " Parties".
2. DESCRIPTION OF PROPERTY
The Parties are each of a residential property located at
(including both the house and the family/live-in at
(including the house and the family/live-in at
3. TERMS. The terms of this Agreement shall be a period of one (1) year, beginning on
(a) the first day of the month of the first year of the Agreement, and ending on the last d

In [ ]:
import torch
from google.colab import userdata
from PIL import Image
import requests
from transformers import pipeline

# 1. Authenticate with Hugging Face using Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Initialize the vision-language pipeline
# You can use "google/paligemma2-3b-mix-224" or larger models like "google/paligemma2-10b-mix-448"
model_id = "google/paligemma2-3b-mix-224"

pipe = pipeline(
    task="image-text-to-text",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

In [ ]:
# 3. Load your image (replace with your own image path or URL)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 4. Standard task prefix for text parsing in PaliGemma is "ocr"
prompt = "describe the image"
# 5. Execute inference
outputs = pipe(image, text=prompt)
print("OCR Result:\n", outputs[0]['generated_text'])


[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OCR Result:
 describe the imageIn this image we can see a car on the road. In the background there is a wall, door and trees.
